In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="max_split_size_mb:256"

In [2]:
import numpy as np
import re
import torch
import random
import sys
import math
import accelerate
from tqdm import tqdm_notebook as tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, LinearLR, SequentialLR
from model_moe import MoETransformer
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
import torch.multiprocessing as mp
import pickle
from contextlib import nullcontext
import gc
import json

In [3]:
!pip show torch accelerate

Name: torch
Version: 2.1.2+cu121
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, triton, typing-extensions
Required-by: accelerate, aimet-torch, clean-fid, clip-anytorch, dctorch, k-diffusion, kornia, lightning, open_clip_torch, pytorch-lightning, qai-hub-models, timm, torchaudio, torchdiffeq, torchmetrics, torchsde, torchvision
---
Name: accelerate
Version: 1.9.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: k-diffusion


In [ ]:
def get_batch(split, accel_device, data_suffix=""):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    device_type = accel_device
    if split == 'train':
        data = np.memmap(os.path.join(data_dir, f'{split}{data_suffix}.bin'), dtype=np.uint16, mode='r')
    else:
        data = np.memmap(os.path.join(data_dir, f'{split}{data_suffix}.bin'), dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
    x, y = x.pin_memory().to(accel_device, non_blocking=True), y.pin_memory().to(accel_device, non_blocking=True)
    return x, y

In [5]:
def get_batch_from_rand_sample(split, accel_device, data_sources=["_owt", "_fweb", "_book"], data_weights = [0.4, 0.4, 0.20] ):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    device_type = accel_device
    if split == 'train':
        data_suffix = random.choices(data_sources, weights=data_weights, k=1)[0]
        data = np.memmap(os.path.join(data_dir, f'{split}{data_suffix}.bin'), dtype=np.uint16, mode='r')
    else:
        data_suffix = data_sources[0]
        data = np.memmap(os.path.join(data_dir, f'{split}{data_suffix}.bin'), dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
    x, y = x.pin_memory().to(accel_device, non_blocking=True), y.pin_memory().to(accel_device, non_blocking=True)
    return x, y

In [6]:
data_dir = './'
model_dir = '/local/mnt2/workspace2/bhargavk/pt_training/model_ckpt_3_31_26'
os.makedirs(model_dir, exist_ok=True)

In [7]:
# Load configurations from config.json
with open('moe_config.json', 'r') as f:
    config = json.load(f)

#model specific
vocab_size = config.get("vocab_size", 50304)
d_model = config.get("d_model", 384)
num_heads = config.get("num_heads", 16)
d_ff = config.get("d_ff", 384 * 2)
num_layers = config.get("num_layers", 16)
num_experts = config.get("num_experts", 16)
batch_size = config.get("batch_size", 4)
block_size = config.get("block_size", 2048)

#training specific
batch_size = config.get("batch_size", 8)
epochs = config.get("epochs", 3)
max_iters = config.get("max_iters", 100)
log_interval = config.get("log_interval", 10)
eval_iters = config.get("eval_iters", 5)
gradient_acc_steps = config.get("gradient_acc_steps", 1)
lr = config.get("lr", 1e-3)
resume_training = config.get("resume_training", False)
load_ckpt = config.get("load_ckpt", "")
ckpt_num = config.get("ckpt_num", 0)
print(config)

{'vocab_size': 50304, 'd_model': 384, 'd_ff': 1024, 'num_layers': 16, 'num_experts': 16, 'block_size': 2048, 'batch_size': 4, 'max_iters': 900000, 'num_heads': 16, 'log_interval': 10000, 'eval_iters': 100, 'gradient_acc_steps': 4, 'lr': 5e-05, 'resume_training': True, 'load_ckpt': './model_ckpt_3_31_26/ckpt_dist_800000.pt', 'ckpt_num': 800000}


In [8]:
model = MoETransformer(vocab_size, d_model, num_heads, d_ff, num_layers, num_experts, max_seq_len=block_size, top_k=2, dropout=0.2)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1, betas=[0.9, 0.95])

#warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=500)
#cos_scheduler = CosineAnnealingLR(optimizer, max_iters-ckpt_num-500, eta_min=1e-5) #ReduceLROnPlateau(optimizer, 'min', patience=3, min_lr=1e-5)
scheduler = CosineAnnealingLR(optimizer, max_iters-ckpt_num, eta_min=1e-5) #SequentialLR(optimizer, schedulers=[warmup, cos_scheduler], milestones=[500])
criterion = torch.nn.CrossEntropyLoss()
mp.set_start_method("spawn", force=True)

/local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


In [9]:
sum([p.numel() for p in model.parameters()])

207728352

In [ ]:
def training_fn_accelerate(model, optimizer, scheduler, max_iters, log_interval, eval_iters, resume_training=False, loaded_weights=None):
    # Create a handler to allow unused parameters (essential for MoE experts)
    ddp_kwargs = accelerate.DistributedDataParallelKwargs(find_unused_parameters=True)
    # Initialize accelerator with the DDP handler and mixed precision
    accelerator = accelerate.Accelerator(
        gradient_accumulation_steps=gradient_acc_steps,
        kwargs_handlers=[ddp_kwargs],
        mixed_precision='bf16'
    )
    
    device = accelerator.device
    
    if resume_training and loaded_weights:
        model.load_state_dict(torch.load(loaded_weights, weights_only=True, map_location='cpu'))
        #optimizer.load_state_dict(torch.load(os.path.join(model_dir, f"optimizer_{ckpt_num}.pt")))
        #print(f"Loaded model and optimizer: {loaded_weights} optimizer_{ckpt_num}")
    # Prepare model, optimizer, and scheduler
    model, scheduler, optimizer = accelerator.prepare(model, scheduler, optimizer)
    
    iter_num = ckpt_num if resume_training else 0   # hardcoded for now, will change later
    model.train()
    recent_loss = 0.0
    recent_steps = 0
    gc.collect()
    torch.cuda.empty_cache()
    pbar = tqdm(total=max_iters, initial=iter_num, disable=not accelerator.is_local_main_process)
    optimizer.zero_grad()
    while iter_num < max_iters:
        with accelerator.accumulate(model):
            X, y = get_batch_from_rand_sample('train', device) #get_batch('train', device, dset_choice)
            
            logits, full_aux_loss = model(X)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            if full_aux_loss is not None:
                loss = loss + 1e-3 * full_aux_loss
            
            accelerator.backward(loss)
            
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            optimizer.zero_grad()

            avg_step_loss = accelerator.gather(loss.detach()).mean().item()
            recent_loss += avg_step_loss
            recent_steps += 1
        if scheduler:
            scheduler.step()
        if (iter_num + 1) % log_interval == 0:
            current_avg_loss = recent_loss / recent_steps
            recent_loss = 0.0
            recent_steps = 0
            @torch.no_grad()
            def estimate_loss_dist(dset_choice):
                model.eval()
                losses = []
                for _ in range(eval_iters):
                    X_v, Y_v = get_batch('val', device, dset_choice)
                    l_out, a_l = model(X_v)
                    v_loss = criterion(l_out.view(-1, l_out.size(-1)), Y_v.view(-1))
                    if a_l is not None: v_loss += 1e-3 * a_l
                    losses.append(v_loss)
                mean_v = torch.stack(losses).mean()
                mean_v = accelerator.gather(mean_v).mean().item()
                model.train()
                return mean_v

            val_loss = estimate_loss_dist("_owt")
            val_loss_book = estimate_loss_dist("_book")
            if accelerator.is_main_process:
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Step {iter_num+1}, Avg Loss: {current_avg_loss:.8f}, Val Losses [OWT+BOOK]: {val_loss:.8f}, {val_loss_book:.8f} LR: {current_lr}')
                
                unwrapped_model = accelerator.unwrap_model(model)
                save_path = os.path.join(model_dir, f'ckpt_dist_{iter_num+1}.pt')
                accelerator.save(unwrapped_model.state_dict(), save_path)
                # accelerator.save(optimizer.state_dict(), os.path.join(model_dir,f'optimizer_{iter_num+1}.pt'))
                with open('training_log_dist.txt', 'a') as f:
                    f.write(f'Step {iter_num+1}, Avg Loss: {current_avg_loss:.8f}, Val Losses [OWT+BOOK]: {val_loss:.8f}, {val_loss_book:.8f} , LR: {current_lr} \n')
    
        iter_num += 1
        if accelerator.is_local_main_process:
            pbar.update(1)
            
    if accelerator.is_main_process:
        pbar.close()
        print('Distributed training complete.')

In [11]:
accelerate.notebook_launcher(training_fn_accelerate, args=(model, optimizer, scheduler, max_iters, log_interval, eval_iters, resume_training, load_ckpt),
                            num_processes=4, mixed_precision="bf16")

Launching training on 4 CUDAs.


/local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/local/mnt2/workspace2/bhargavk/venv_310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in th

 89%|########8 | 800000/900000 [00:00<?, ?it/s]

[W reducer.cpp:1346] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
[W reducer.cpp:1346] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later 

Step 810000, Avg Loss: 3.45149623, Val Losses [OWT+BOOK]: 3.41073418, 3.47197247 LR: 4.902113032590287e-05
Step 820000, Avg Loss: 3.37934200, Val Losses [OWT+BOOK]: 3.34358096, 3.39485049 LR: 4.618033988749868e-05
Step 830000, Avg Loss: 3.35650216, Val Losses [OWT+BOOK]: 3.33657980, 3.40147972 LR: 4.175570504584911e-05
Step 840000, Avg Loss: 3.33492512, Val Losses [OWT+BOOK]: 3.32035732, 3.38453650 LR: 3.618033988751032e-05
Step 850000, Avg Loss: 3.32250959, Val Losses [OWT+BOOK]: 3.30666208, 3.33205175 LR: 3.0000000000023518e-05
Step 860000, Avg Loss: 3.31643137, Val Losses [OWT+BOOK]: 3.31193972, 3.34583569 LR: 2.3819660112527452e-05
Step 870000, Avg Loss: 3.31222552, Val Losses [OWT+BOOK]: 3.28585577, 3.32934546 LR: 1.8244294954166238e-05
Step 880000, Avg Loss: 3.30216748, Val Losses [OWT+BOOK]: 3.27056074, 3.31026149 LR: 1.3819660112508374e-05
Step 890000, Avg Loss: 3.29640261, Val Losses [OWT+BOOK]: 3.25646687, 3.30678105 LR: 1.0978869674098802e-05
Step 900000, Avg Loss: 3.2950011

[2026-04-09 20:37:59,103] torch.distributed.elastic.multiprocessing.api: [WARNING] Closing process 3623064 via signal SIGTERM
[2026-04-09 20:37:59,106] torch.distributed.elastic.multiprocessing.api: [WARNING] Closing process 3623066 via signal SIGTERM
[2026-04-09 20:37:59,107] torch.distributed.elastic.multiprocessing.api: [WARNING] Closing process 3623068 via signal SIGTERM
